# PDF to MCQ Generator
Step-by-step walkthrough of the full pipeline.

## 1. Imports

In [1]:
import os
import io
import json
import re
import shutil
from pathlib import Path
from PyPDF2 import PdfReader

try:
    from pdf2image import convert_from_bytes
    import pytesseract
    OCR_AVAILABLE = True
except ImportError:
    OCR_AVAILABLE = False

print('OCR libraries available:', OCR_AVAILABLE)
print('tesseract binary:', shutil.which('tesseract'))
print('pdftoppm binary:', shutil.which('pdftoppm'))

OCR libraries available: True
tesseract binary: /usr/bin/tesseract
pdftoppm binary: /usr/bin/pdftoppm


## 2. Load Environment Variables

In [2]:
from dotenv import load_dotenv

env_path = Path('..') / '.env'
load_dotenv(env_path)

GROQ_API_KEY = os.getenv('GROQ_API_KEY')
print('GROQ_API_KEY loaded:', bool(GROQ_API_KEY))

GROQ_API_KEY loaded: True


## 3. Set PDF Path

In [3]:
# Change this to your PDF file path
PDF_PATH = Path('../..') / 'sample.pdf'

print('File exists:', PDF_PATH.exists())
print('File size:', PDF_PATH.stat().st_size, 'bytes')

File exists: True
File size: 192123 bytes


## 4. Extract Text with PyPDF2

In [4]:
pdf_bytes = PDF_PATH.read_bytes()
reader = PdfReader(io.BytesIO(pdf_bytes))

print('Total pages:', len(reader.pages))

text = ''
for i, page in enumerate(reader.pages):
    page_text = page.extract_text() or ''
    text += page_text + '\n'
    print(f'  Page {i+1}: {len(page_text)} chars')

text = text.strip()
print('\nTotal extracted chars:', len(text))

Total pages: 2
  Page 1: 0 chars
  Page 2: 0 chars

Total extracted chars: 0


## 5. OCR Fallback (for Image-Based PDFs)

In [5]:
if not text:
    print('No text from PyPDF2 — trying OCR...')

    if not OCR_AVAILABLE:
        raise RuntimeError('pdf2image / pytesseract not installed')
    if not shutil.which('tesseract') or not shutil.which('pdftoppm'):
        raise RuntimeError('System tools missing: install tesseract-ocr and poppler-utils')

    images = convert_from_bytes(pdf_bytes)
    ocr_parts = []
    for i, img in enumerate(images, 1):
        page_text = pytesseract.image_to_string(img).strip()
        print(f'  OCR page {i}: {len(page_text)} chars')
        if page_text:
            ocr_parts.append(page_text)

    text = '\n'.join(ocr_parts).strip()
    print('\nTotal OCR chars:', len(text))
else:
    print('Text extracted via PyPDF2 — OCR not needed.')

No text from PyPDF2 — trying OCR...
  OCR page 1: 1254 chars
  OCR page 2: 459 chars

Total OCR chars: 1714


## 6. Preview Extracted Text

In [6]:
print(text[:1000])

&g» Webpack - Ideas List
wo

Webpack: Universal Targets

Difficulty: Hard
Project Size: 350 hours
Mentors: Alexander Akait, Nitin Kumar

Introduction

This project aims to enhance the versatility of Webpack's targets. Currently,
there are limitations, and a web bundle doesn't seamlessly fit into Node.js or a
web worker environment. The proposal is to introduce a Universal Target that
incorporates runtime code suitable for web, web worker, and Node.js. While this
may increase code size slightly, the benefit is the ability to create Universal
Module Definition (UMD) bundles that work seamlessly across all environments.
Additionally, this enhancement facilitates sharing chunks between web and web
worker environments.

Related Issue(s): #6525

Prerequisites
JavaScript, Node.js

Webpack: New Documentation Website

Difficulty: Medium
Project Size: 350 hours
Mentors: Claudio Wunder, Aviv Keller, Sebastian Beltran

Introduction
We’re redesigning and reworking on Webpack’s website and its docum

## 7. Build the MCQ Prompt

In [7]:
NUM_QUESTIONS = 5

# Truncate if too long
input_text = text[:10000] + '...(truncated)' if len(text) > 10000 else text

prompt = f"""From this text, make {NUM_QUESTIONS} multiple choice questions:

TEXT:
{input_text}

Return JSON only:
{{
    "questions": [
        {{
            "question": "Question?",
            "options": ["A", "B", "C", "D"],
            "answer": "A"
        }}
    ]
}}

Rules: Exactly {NUM_QUESTIONS} questions, 4 options each, JSON only."""

print('Prompt length:', len(prompt), 'chars')

Prompt length: 2003 chars


## 8. Call Groq API

In [8]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

response = client.chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant that generates multiple choice questions.'},
        {'role': 'user', 'content': prompt}
    ],
    temperature=0.7,
    max_tokens=4000
)

raw = response.choices[0].message.content
print('Raw response length:', len(raw), 'chars')
print(raw[:300])

Raw response length: 1422 chars
```
{
    "questions": [
        {
            "question": "What is the main goal of the Webpack Universal Targets project?",
            "options": ["To reduce code size", "To enhance the versatility of Webpack's targets", "To create a new documentation website", "To add support for CSS"],
        


## 9. Parse the Response

In [9]:
def parse_mcqs(resp):
    resp = resp.strip()
    # Strip markdown code fences
    resp = re.sub(r'^```json\s*', '', resp)
    resp = re.sub(r'^```\s*', '', resp)
    resp = re.sub(r'\s*```$', '', resp)

    try:
        return json.loads(resp)['questions']
    except (json.JSONDecodeError, KeyError):
        pass

    # Fallback: find JSON object anywhere in string
    match = re.search(r'\{[\s\S]*\}', resp)
    if match:
        return json.loads(match.group(0))['questions']

    raise ValueError('Could not parse MCQ response')


questions = parse_mcqs(raw)
print(f'Parsed {len(questions)} questions')

Parsed 5 questions


## 10. Display Results

In [ ]:
for i, q in enumerate(questions, 1):
    print(f'Q{i}: {q["question"]}')
    for opt in q['options']:
        print(f'   {opt}')
    print(f'   Answer: {q["answer"]}')
    print()

Q1: What is the main goal of the Webpack Universal Targets project?
   To reduce code size
   To enhance the versatility of Webpack's targets
   To create a new documentation website
   To add support for CSS
   Answer: B

Q2: What is the estimated project size for Webpack Universal Targets?
   100 hours
   200 hours
   350 hours
   500 hours
   Answer: C

Q3: What are the prerequisites for the Webpack New Documentation Website project?
   JavaScript only
   Node.js only
   JavaScript, Node.js
   JavaScript, CSS, HTML
   Answer: D

Q4: What is the main benefit of introducing a Universal Target in Webpack?
   Reduced code size
   Improved performance
   Ability to create Universal Module Definition (UMD) bundles
   Simplified debugging
   Answer: C

Q5: What type of files does the Webpack Entry points as HTML project aim to support as entry points?
   JavaScript files only
   HTML files only
   CSS files only
   HTML files in addition to JavaScript files
   Answer: D



Bad pipe message: %s [b'0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7\r\nHost: localhost:34325\r\nUs', b'-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.']
Bad pipe message: %s [b'0.0 Safari/537.36\r\nAccept-Encoding: gzip, defla']
Bad pipe message: %s [b', br, zstd\r\nAccept-Language: en-US,en;q=0.9\r\nCache-Control: max-age=0\r\nReferer: https://github.com/\r\nX-Request-ID: ', b'626311802c53e16d5c45728d3071d1\r\nX-Real-IP: 103.190.7.39\r', b'-Forwarde']
Bad pipe message: %s [b'Port: 443\r\nX-Forwarded-Scheme: https\r\nX-Original-URI: /\r\nX-Scheme: https\r\nsec-fetch-site: cross-sit', b'\nsec-fetch-mode: navigate\r\nsec-fetch-dest: document\r\nsec-ch-ua: "Not:A-Brand";v="99", "Google Chrome', b'v="145", "Chromium";v="145"\r\nsec-', b'-ua-mobile: ?0\r\nsec-ch-ua-platform: "Windows"\r\npriority: u=0, i\r\nCookie: _ga_9SCKP7M3T5=GS2.1.s177']
Bad pipe message: %s [b'80903$o9$g1$t1772387116$j60$l0$